In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Import system components
from soros_system.main import TrendAnalyzer
from soros_system.analysis.markov_vol_model import MarkovVolModel
from soros_system.analysis.forward_returns.signal_evaluator import SignalEvaluator
from soros_system.analysis.forward_returns.statistical_tests import StatisticalTester
from soros_system.portfolio.signal_selector import SignalSelector

# Import all signal classes
from soros_system.signals.trend_signals import (
    # USD Quote Trend Signals
    ShortTermStrongBearUSD, ShortTermWeakBearUSD, ShortTermNeutralUSD, ShortTermWeakBullUSD, ShortTermStrongBullUSD,
    MediumTermStrongBearUSD, MediumTermWeakBearUSD, MediumTermNeutralUSD, MediumTermWeakBullUSD, MediumTermStrongBullUSD,
    LongTermStrongBearUSD, LongTermWeakBearUSD, LongTermNeutralUSD, LongTermWeakBullUSD, LongTermStrongBullUSD,
    OverallStrongBearUSD, OverallWeakBearUSD, OverallNeutralUSD, OverallWeakBullUSD, OverallStrongBullUSD,
    # BTC Quote Trend Signals
    ShortTermStrongBearBTC, ShortTermWeakBearBTC, ShortTermNeutralBTC, ShortTermWeakBullBTC, ShortTermStrongBullBTC,
    MediumTermStrongBearBTC, MediumTermWeakBearBTC, MediumTermNeutralBTC, MediumTermWeakBullBTC, MediumTermStrongBullBTC,
    LongTermStrongBearBTC, LongTermWeakBearBTC, LongTermNeutralBTC, LongTermWeakBullBTC, LongTermStrongBullBTC,
    OverallStrongBearBTC, OverallWeakBearBTC, OverallNeutralBTC, OverallWeakBullBTC, OverallStrongBullBTC
)
from soros_system.signals.rsi_signals import (
    RSI_Oversold_USD, RSI_Overbought_USD, RSI_Bullish_USD, RSI_Bearish_USD,
    RSI_Oversold_BTC, RSI_Overbought_BTC, RSI_Bullish_BTC, RSI_Bearish_BTC
)
from soros_system.signals.volatility_signals import (
    MarkovLowVolatilitySignal, MarkovHighVolatilitySignal
)
from soros_system.signals.ssr_signals import SSR_RiskOn, SSR_RiskOff

# Set paths and initialize analyzer
data_path='/Users/valter.rebelo/MissionControl/data/micro/candleData/'
btc_data_path='/Users/valter.rebelo/MissionControl/data/micro/candleData/bitcoin_candles.csv'
ssr_data_path='/Users/valter.rebelo/MissionControl/data/onchainData/BTC_SSR.csv'
market_data_path="/Users/valter.rebelo/MissionControl/data/micro/assetData/"
markov_vol_model = MarkovVolModel.load_model_by_timestamp('20250325_144631')

# Initialize analyzer with test assets
test_assets = ['bitcoin', 'ethereum', 'solana', 'chainlink', 'aave']
print(f"Testing with assets: {test_assets}")

analyzer = TrendAnalyzer(
    asset_ids=test_assets,
    data_path=data_path,0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
        ..
3523   NaN
3524   NaN
3525   NaN
3526   NaN
3527   NaN
Length: 3528, dtype: float64
Signal active on 0 dates
Signal weight: 0.0
Optimal holding period: 0 days
Is effective: False
    btc_data_path=btc_data_path,
    lookback_days='all',
    ssr_data_path=ssr_data_path,        
    markov_analyzer=markov_vol_model,
    market_data_path=market_data_path  
)

# Run analysis
analyzer.analyze_multiple_assets()

# Get data for Ethereum
eth_data = analyzer.get_asset_raw_data('ethereum')
print(f"\nEthereum data available from {eth_data['date'].min()} to {eth_data['date'].max()}")
print(f"Total data points: {len(eth_data)}")

In [2]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Copy the essential fixed methods from forward returns calculator
class BinaryForwardReturnsCalculator:
    """Fixed calculator that works with binary (0/1) signals instead of (-1/1)."""
    
    def __init__(self, periods=None):
        self.periods = periods or [3, 5, 7, 14, 30, 60]
    
    def calculate_forward_returns(self, data, price_col='close'):
        """Calculate forward returns for multiple periods."""
        df = data.copy()
        
        # Ensure data is sorted by date
        if 'date' in df.columns:
            df = df.sort_values('date')
        
        # Calculate forward returns for each period
        for period in self.periods:
            col_name = f'fwd_ret_{period}d'
            try:
                df[col_name] = np.nan
                for i in range(len(df) - period):
                    df.iloc[i, df.columns.get_loc(col_name)] = (
                        df.iloc[i + period][price_col] / df.iloc[i][price_col]
                    ) - 1
            except Exception as e:
                print(f"Error calculating {period}-day forward returns: {e}")
                df[col_name] = np.nan
        
        return df
    
    def get_conditional_returns(self, data, signal_series, min_samples=20):
        """Get forward returns conditioned on signal values (FIXED for 0/1 signals)."""
        df = data.copy()
        
        # Add signal column
        df['signal'] = signal_series.values
        
        # Get forward return columns
        fwd_cols = [col for col in df.columns if col.startswith('fwd_ret_')]
        
        # Initialize dictionaries
        conditional_returns = {}
        
        # Filter returns based on signal value
        # KEY FIX: Use 1 for positive and 0 for negative instead of 1/-1
        positive_signal = df[df['signal'] == 1]  # Signal is active
        negative_signal = df[df['signal'] == 0]  # Signal is inactive 
        all_signal = df
        
        # Calculate sample counts
        sample_counts = {
            'positive': len(positive_signal),
            'negative': len(negative_signal),
            'all': len(all_signal)
        }
        
        print(f"Signal distribution: {sample_counts['positive']} positive (1), "
              f"{sample_counts['negative']} negative (0), {sample_counts['all']} total")
        
        # Check if we have sufficient samples
        has_positive = sample_counts['positive'] >= min_samples
        has_negative = sample_counts['negative'] >= min_samples
        
        if not has_positive and not has_negative:
            print(f"Insufficient samples for both signal states. Need at least {min_samples}.")
        elif not has_positive:
            print(f"Insufficient samples for positive signal. Got {sample_counts['positive']}.")
        elif not has_negative:
            print(f"Insufficient samples for negative signal. Got {sample_counts['negative']}.")
        
        # Extract conditional returns for each period
        for col in fwd_cols:
            period = int(col.split('_')[2].replace('d', ''))
            period_dict = {}
            
            if has_positive:
                period_dict['positive'] = positive_signal[col].dropna()
            
            if has_negative:
                period_dict['negative'] = negative_signal[col].dropna()
            
            period_dict['all'] = all_signal[col].dropna()
            conditional_returns[period] = period_dict
        
        return conditional_returns, sample_counts

In [ ]:
# Comprehensive analysis of a single signal
asset = 'aave'

# Get data for Ethereum
eth_data = analyzer.get_asset_raw_data(asset)
print(f"Testing with {len(eth_data)} days of {asset} data")

# Create instances of RSI signals for USD
rsi_oversold = RSI_Oversold_USD(params={'rsi_length': 14, 'oversold': 30})
rsi_overbought = RSI_Overbought_USD(params={'rsi_length': 14, 'overbought': 70})
rsi_bullish = RSI_Bullish_USD(params={'rsi_length': 28})
rsi_bearish = RSI_Bearish_USD(params={'rsi_length': 28})

# Create instances of RSI signals for BTC
rsi_oversold_btc = RSI_Oversold_BTC(params={'rsi_length': 14, 'oversold': 30})
rsi_overbought_btc = RSI_Overbought_BTC(params={'rsi_length': 14, 'overbought': 70})
rsi_bullish_btc = RSI_Bullish_BTC(params={'rsi_length': 28})
rsi_bearish_btc = RSI_Bearish_BTC(params={'rsi_length': 28})

# Calculate the USD signals
oversold_values = rsi_oversold.calculate(eth_data, asset)
overbought_values = rsi_overbought.calculate(eth_data, asset)
bullish_values = rsi_bullish.calculate(eth_data, asset)
bearish_values = rsi_bearish.calculate(eth_data, asset)

# Calculate the BTC signals
oversold_btc_values = rsi_oversold_btc.calculate(eth_data, asset)
overbought_btc_values = rsi_overbought_btc.calculate(eth_data, asset)
bullish_btc_values = rsi_bullish_btc.calculate(eth_data, asset)
bearish_btc_values = rsi_bearish_btc.calculate(eth_data, asset)

print(f"Oversold USD signal values: 1s: {sum(oversold_values == 1)}, 0s: {sum(oversold_values == 0)}")
print(f"Overbought USD signal values: 1s: {sum(overbought_values == 1)}, 0s: {sum(overbought_values == 0)}")
print(f"Bullish USD signal values: 1s: {sum(bullish_values == 1)}, 0s: {sum(bullish_values == 0)}")
print(f"Bearish USD signal values: 1s: {sum(bearish_values == 1)}, 0s: {sum(bearish_values == 0)}")

print(f"Oversold BTC signal values: 1s: {sum(oversold_btc_values == 1)}, 0s: {sum(oversold_btc_values == 0)}")
print(f"Overbought BTC signal values: 1s: {sum(overbought_btc_values == 1)}, 0s: {sum(overbought_btc_values == 0)}")
print(f"Bullish BTC signal values: 1s: {sum(bullish_btc_values == 1)}, 0s: {sum(bullish_btc_values == 0)}")
print(f"Bearish BTC signal values: 1s: {sum(bearish_btc_values == 1)}, 0s: {sum(bearish_btc_values == 0)}")

# Create our fixed calculator
calculator = BinaryForwardReturnsCalculator(periods=[3, 5, 7, 14, 30, 60])

# Calculate forward returns
data_with_returns = calculator.calculate_forward_returns(eth_data, 'close')

# Set up statistical tester
from soros_system.analysis.forward_returns.statistical_tests import StatisticalTester
tester = StatisticalTester(alpha=0.05)

# Analyze all signals
for signal_name, signal_values in [
    ("RSI Oversold USD", oversold_values), 
    ("RSI Overbought USD", overbought_values), 
    ("RSI Bullish USD", bullish_values), 
    ("RSI Bearish USD", bearish_values),
    ("RSI Oversold BTC", oversold_btc_values),
    ("RSI Overbought BTC", overbought_btc_values),
    ("RSI Bullish BTC", bullish_btc_values),
    ("RSI Bearish BTC", bearish_btc_values)
]:
    print(f"\n\n{'='*50}")
    print(f"ANALYZING {signal_name} SIGNAL")
    print(f"{'='*50}")
    
    # Get conditional returns using our fixed method that understands 0/1 signals
    conditional_returns, sample_counts = calculator.get_conditional_returns(
        data_with_returns, signal_values, min_samples=30
    )
    
    # Check if we have sufficient samples
    if sample_counts['positive'] >= 30 and sample_counts['negative'] >= 30:
        print("We have enough samples for analysis!")
        
        # Initialize results storage
        all_test_results = {}
        
        # Analyze each period
        for period in calculator.periods:
            if period in conditional_returns and 'positive' in conditional_returns[period] and 'negative' in conditional_returns[period]:
                positive_returns = conditional_returns[period]['positive']
                negative_returns = conditional_returns[period]['negative']
                all_returns = conditional_returns[period]['all']
                
                print(f"\n=== {period}-Day Forward Returns Analysis ===")
                print(f"Signal=1 samples: {len(positive_returns)}, Signal=0 samples: {len(negative_returns)}")
                
                # Run all statistical tests
                test_results = tester.run_all_tests(positive_returns, negative_returns, all_returns)
                all_test_results[period] = test_results
                
                # Evaluate effectiveness
                effectiveness = tester.evaluate_signal_effectiveness(test_results)
                
                # Print results summary
                print(f"\nEffectiveness Summary for {period}-day period:")
                print(f"Overall effective: {effectiveness['overall_effective']}")
                print(f"Confidence: {effectiveness['confidence']:.4f}")
                print(f"Effect size: {effectiveness['effect_size']:.4f}")
                
                # T-test details
                if 't_test' in test_results and test_results['t_test'].get('valid', False):
                    t_test = test_results['t_test']
                    print(f"\nT-test (mean comparison):")
                    print(f"  Significant: {t_test.get('significant', False)}")
                    print(f"  P-value: {t_test.get('p_value', 'N/A'):.4f}")
                    print(f"  Mean difference: {t_test.get('mean_difference', 0):.4f}")
                    print(f"  Effect size: {t_test.get('effect_size', 0):.4f}")
                    
                # Mann-Whitney details  
                if 'mann_whitney' in test_results and test_results['mann_whitney'].get('valid', False):
                    mw_test = test_results['mann_whitney']
                    print(f"\nMann-Whitney (distribution comparison):")
                    print(f"  Significant: {mw_test.get('significant', False)}")
                    print(f"  P-value: {mw_test.get('p_value', 'N/A'):.4f}")
                    print(f"  Median difference: {mw_test.get('median_difference', 0):.4f}")
                
                # KS-test details
                if 'ks_test' in test_results and test_results['ks_test'].get('valid', False):
                    ks_test = test_results['ks_test']
                    print(f"\nKolmogorov-Smirnov (distribution shape):")
                    print(f"  Significant: {ks_test.get('significant', False)}")
                    print(f"  P-value: {ks_test.get('p_value', 'N/A'):.4f}")
                    print(f"  Statistic: {ks_test.get('statistic', 0):.4f}")
                
                # Skew & Kurt details
                if 'skew_kurt' in test_results and test_results['skew_kurt'].get('valid', False):
                    sk_test = test_results['skew_kurt']
                    print(f"\nSkewness & Kurtosis:")
                    print(f"  Signal=1 skew: {sk_test.get('positive_skew', 0):.4f}")
                    print(f"  Signal=0 skew: {sk_test.get('negative_skew', 0):.4f}")
                    print(f"  Signal=1 kurt: {sk_test.get('positive_kurt', 0):.4f}")
                    print(f"  Signal=0 kurt: {sk_test.get('negative_kurt', 0):.4f}")
        
        # Calculate overall effectiveness
        effective_periods = 0
        total_periods = len(all_test_results)
        effect_sizes = []
        confidences = []
        mean_diffs = []
        
        for period, test_results in all_test_results.items():
            effectiveness = tester.evaluate_signal_effectiveness(test_results)
            if effectiveness.get('overall_effective', False):
                effective_periods += 1
                effect_sizes.append(abs(effectiveness.get('effect_size', 0.0)))
                confidences.append(effectiveness.get('confidence', 0.0))
                
                if 't_test' in test_results and test_results['t_test'].get('valid', False):
                    mean_diffs.append(test_results['t_test'].get('mean_difference', 0))
        
        # Calculate summary metrics
        effectiveness_ratio = effective_periods / total_periods if total_periods > 0 else 0.0
        avg_effect_size = np.mean(effect_sizes) if effect_sizes else 0.0
        avg_confidence = np.mean(confidences) if confidences else 0.0
        avg_mean_diff = np.mean(mean_diffs) if mean_diffs else 0.0
        
        # Determine overall effectiveness
        overall_effective = effectiveness_ratio >= 0.5 and avg_effect_size > 0.15
        
        # Calculate weight
        if overall_effective:
            weight = (avg_effect_size * 0.5) + (avg_confidence * 0.3) + (abs(avg_mean_diff) * 100 * 0.2)
        else:
            weight = (avg_effect_size * 0.25) + (avg_confidence * 0.15) + (abs(avg_mean_diff) * 100 * 0.1)
            if effectiveness_ratio < 0.3 or avg_effect_size < 0.1:
                weight = 0.0
        
        # Print overall summary
        signal_obj = None
        if signal_name == "RSI Oversold USD":
            signal_obj = rsi_oversold
        elif signal_name == "RSI Overbought USD":
            signal_obj = rsi_overbought
        elif signal_name == "RSI Bullish USD":
            signal_obj = rsi_bullish
        elif signal_name == "RSI Bearish USD":
            signal_obj = rsi_bearish
        elif signal_name == "RSI Oversold BTC":
            signal_obj = rsi_oversold_btc
        elif signal_name == "RSI Overbought BTC":
            signal_obj = rsi_overbought_btc
        elif signal_name == "RSI Bullish BTC":
            signal_obj = rsi_bullish_btc
        elif signal_name == "RSI Bearish BTC":
            signal_obj = rsi_bearish_btc
            
        print("\n=== OVERALL SUMMARY ===")
        print(f"Signal: {signal_obj.name}")
        print(f"Effective periods: {effective_periods}/{total_periods}")
        print(f"Effectiveness ratio: {effectiveness_ratio:.4f}")
        print(f"Average effect size: {avg_effect_size:.4f}")
        print(f"Average confidence: {avg_confidence:.4f}")
        print(f"Average mean difference: {avg_mean_diff:.4f}")
        print(f"Overall effectiveness: {overall_effective}")
        print(f"Weight: {weight:.4f}")
        
    else:
        print("Not enough samples for analysis")

# Now generate all the plots after all the analysis is printed
for signal_name, signal_values in [
    ("RSI Oversold USD", oversold_values), 
    ("RSI Overbought USD", overbought_values), 
    ("RSI Bullish USD", bullish_values), 
    ("RSI Bearish USD", bearish_values),
    ("RSI Oversold BTC", oversold_btc_values),
    ("RSI Overbought BTC", overbought_btc_values),
    ("RSI Bullish BTC", bullish_btc_values),
    ("RSI Bearish BTC", bearish_btc_values)
]:
    print(f"\n\n{'='*50}")
    print(f"GENERATING PLOTS FOR {signal_name} SIGNAL")
    print(f"{'='*50}")
    
    # Get conditional returns
    conditional_returns, sample_counts = calculator.get_conditional_returns(
        data_with_returns, signal_values, min_samples=30
    )
    
    # Check if we have sufficient samples
    if sample_counts['positive'] >= 30 and sample_counts['negative'] >= 30:
        # Analyze each period
        for period in calculator.periods:
            if period in conditional_returns and 'positive' in conditional_returns[period] and 'negative' in conditional_returns[period]:
                positive_returns = conditional_returns[period]['positive']
                negative_returns = conditional_returns[period]['negative']
                
                # Plot distributions
                plt.figure(figsize=(12, 6))
                sns.kdeplot(data=positive_returns, label='Signal = 1', color='green', alpha=0.6)
                sns.kdeplot(data=negative_returns, label='Signal = 0', color='red', alpha=0.6)
                
                # Add vertical lines for means and medians
                plt.axvline(positive_returns.mean(), color='green', linestyle='--', alpha=0.8, label='Mean (Signal=1)')
                plt.axvline(negative_returns.mean(), color='red', linestyle='--', alpha=0.8, label='Mean (Signal=0)')
                plt.axvline(positive_returns.median(), color='green', linestyle=':', alpha=0.8, label='Median (Signal=1)')
                plt.axvline(negative_returns.median(), color='red', linestyle=':', alpha=0.8, label='Median (Signal=0)')
                
                # Add title and labels
                signal_obj = None
                if signal_name == "RSI Oversold USD":
                    signal_obj = rsi_oversold
                elif signal_name == "RSI Overbought USD":
                    signal_obj = rsi_overbought
                elif signal_name == "RSI Bullish USD":
                    signal_obj = rsi_bullish
                elif signal_name == "RSI Bearish USD":
                    signal_obj = rsi_bearish
                elif signal_name == "RSI Oversold BTC":
                    signal_obj = rsi_oversold_btc
                elif signal_name == "RSI Overbought BTC":
                    signal_obj = rsi_overbought_btc
                elif signal_name == "RSI Bullish BTC":
                    signal_obj = rsi_bullish_btc
                elif signal_name == "RSI Bearish BTC":
                    signal_obj = rsi_bearish_btc
                    
                plt.title(f'{signal_obj.name} - {period}-Day Forward Return Distributions')
                plt.xlabel('Forward Returns')
                plt.ylabel('Density')
                plt.legend()
                
                # Add statistical annotations
                stats_text = f"""
                Signal=1 Mean: {positive_returns.mean():.4f}
                Signal=0 Mean: {negative_returns.mean():.4f}
                Signal=1 Median: {positive_returns.median():.4f}
                Signal=0 Median: {negative_returns.median():.4f}
                Signal=1 Std: {positive_returns.std():.4f}
                Signal=0 Std: {negative_returns.std():.4f}
                Signal=1 Skew: {stats.skew(positive_returns):.4f}
                Signal=0 Skew: {stats.skew(negative_returns):.4f}
                Signal=1 Count: {len(positive_returns)}
                Signal=0 Count: {len(negative_returns)}
                """
                plt.text(0.02, 0.98, stats_text,
                         transform=plt.gca().transAxes,
                         verticalalignment='top',
                         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                
                plt.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()
    else:
        print("Not enough samples for plots")

In [ ]:
# Comprehensive analysis of a single signal
asset = 'aave'

# Get data for Ethereum
eth_data = analyzer.get_asset_raw_data(asset)
print(f"Testing with {len(eth_data)} days of {asset} data")

# Create instances of USD Quote Trend Signals
short_term_strong_bear_usd = ShortTermStrongBearUSD()
short_term_weak_bear_usd = ShortTermWeakBearUSD()
short_term_neutral_usd = ShortTermNeutralUSD()
short_term_weak_bull_usd = ShortTermWeakBullUSD()
short_term_strong_bull_usd = ShortTermStrongBullUSD()

medium_term_strong_bear_usd = MediumTermStrongBearUSD()
medium_term_weak_bear_usd = MediumTermWeakBearUSD()
medium_term_neutral_usd = MediumTermNeutralUSD()
medium_term_weak_bull_usd = MediumTermWeakBullUSD()
medium_term_strong_bull_usd = MediumTermStrongBullUSD()

long_term_strong_bear_usd = LongTermStrongBearUSD()
long_term_weak_bear_usd = LongTermWeakBearUSD()
long_term_neutral_usd = LongTermNeutralUSD()
long_term_weak_bull_usd = LongTermWeakBullUSD()
long_term_strong_bull_usd = LongTermStrongBullUSD()

overall_strong_bear_usd = OverallStrongBearUSD()
overall_weak_bear_usd = OverallWeakBearUSD()
overall_neutral_usd = OverallNeutralUSD()
overall_weak_bull_usd = OverallWeakBullUSD()
overall_strong_bull_usd = OverallStrongBullUSD()

# Create instances of BTC Quote Trend Signals
short_term_strong_bear_btc = ShortTermStrongBearBTC()
short_term_weak_bear_btc = ShortTermWeakBearBTC()
short_term_neutral_btc = ShortTermNeutralBTC()
short_term_weak_bull_btc = ShortTermWeakBullBTC()
short_term_strong_bull_btc = ShortTermStrongBullBTC()

medium_term_strong_bear_btc = MediumTermStrongBearBTC()
medium_term_weak_bear_btc = MediumTermWeakBearBTC()
medium_term_neutral_btc = MediumTermNeutralBTC()
medium_term_weak_bull_btc = MediumTermWeakBullBTC()
medium_term_strong_bull_btc = MediumTermStrongBullBTC()

long_term_strong_bear_btc = LongTermStrongBearBTC()
long_term_weak_bear_btc = LongTermWeakBearBTC()
long_term_neutral_btc = LongTermNeutralBTC()
long_term_weak_bull_btc = LongTermWeakBullBTC()
long_term_strong_bull_btc = LongTermStrongBullBTC()

overall_strong_bear_btc = OverallStrongBearBTC()
overall_weak_bear_btc = OverallWeakBearBTC()
overall_neutral_btc = OverallNeutralBTC()
overall_weak_bull_btc = OverallWeakBullBTC()
overall_strong_bull_btc = OverallStrongBullBTC()

# Calculate the USD signals
short_term_strong_bear_usd_values = short_term_strong_bear_usd.calculate(eth_data, asset)
short_term_weak_bear_usd_values = short_term_weak_bear_usd.calculate(eth_data, asset)
short_term_neutral_usd_values = short_term_neutral_usd.calculate(eth_data, asset)
short_term_weak_bull_usd_values = short_term_weak_bull_usd.calculate(eth_data, asset)
short_term_strong_bull_usd_values = short_term_strong_bull_usd.calculate(eth_data, asset)

medium_term_strong_bear_usd_values = medium_term_strong_bear_usd.calculate(eth_data, asset)
medium_term_weak_bear_usd_values = medium_term_weak_bear_usd.calculate(eth_data, asset)
medium_term_neutral_usd_values = medium_term_neutral_usd.calculate(eth_data, asset)
medium_term_weak_bull_usd_values = medium_term_weak_bull_usd.calculate(eth_data, asset)
medium_term_strong_bull_usd_values = medium_term_strong_bull_usd.calculate(eth_data, asset)

long_term_strong_bear_usd_values = long_term_strong_bear_usd.calculate(eth_data, asset)
long_term_weak_bear_usd_values = long_term_weak_bear_usd.calculate(eth_data, asset)
long_term_neutral_usd_values = long_term_neutral_usd.calculate(eth_data, asset)
long_term_weak_bull_usd_values = long_term_weak_bull_usd.calculate(eth_data, asset)
long_term_strong_bull_usd_values = long_term_strong_bull_usd.calculate(eth_data, asset)

overall_strong_bear_usd_values = overall_strong_bear_usd.calculate(eth_data, asset)
overall_weak_bear_usd_values = overall_weak_bear_usd.calculate(eth_data, asset)
overall_neutral_usd_values = overall_neutral_usd.calculate(eth_data, asset)
overall_weak_bull_usd_values = overall_weak_bull_usd.calculate(eth_data, asset)
overall_strong_bull_usd_values = overall_strong_bull_usd.calculate(eth_data, asset)

# Calculate the BTC signals
short_term_strong_bear_btc_values = short_term_strong_bear_btc.calculate(eth_data, asset)
short_term_weak_bear_btc_values = short_term_weak_bear_btc.calculate(eth_data, asset)
short_term_neutral_btc_values = short_term_neutral_btc.calculate(eth_data, asset)
short_term_weak_bull_btc_values = short_term_weak_bull_btc.calculate(eth_data, asset)
short_term_strong_bull_btc_values = short_term_strong_bull_btc.calculate(eth_data, asset)

medium_term_strong_bear_btc_values = medium_term_strong_bear_btc.calculate(eth_data, asset)
medium_term_weak_bear_btc_values = medium_term_weak_bear_btc.calculate(eth_data, asset)
medium_term_neutral_btc_values = medium_term_neutral_btc.calculate(eth_data, asset)
medium_term_weak_bull_btc_values = medium_term_weak_bull_btc.calculate(eth_data, asset)
medium_term_strong_bull_btc_values = medium_term_strong_bull_btc.calculate(eth_data, asset)

long_term_strong_bear_btc_values = long_term_strong_bear_btc.calculate(eth_data, asset)
long_term_weak_bear_btc_values = long_term_weak_bear_btc.calculate(eth_data, asset)
long_term_neutral_btc_values = long_term_neutral_btc.calculate(eth_data, asset)
long_term_weak_bull_btc_values = long_term_weak_bull_btc.calculate(eth_data, asset)
long_term_strong_bull_btc_values = long_term_strong_bull_btc.calculate(eth_data, asset)

overall_strong_bear_btc_values = overall_strong_bear_btc.calculate(eth_data, asset)
overall_weak_bear_btc_values = overall_weak_bear_btc.calculate(eth_data, asset)
overall_neutral_btc_values = overall_neutral_btc.calculate(eth_data, asset)
overall_weak_bull_btc_values = overall_weak_bull_btc.calculate(eth_data, asset)
overall_strong_bull_btc_values = overall_strong_bull_btc.calculate(eth_data, asset)

# Print signal counts
print("\nUSD Quote Trend Signals:")
print(f"Short Term Strong Bear USD: 1s: {sum(short_term_strong_bear_usd_values == 1)}, 0s: {sum(short_term_strong_bear_usd_values == 0)}")
print(f"Short Term Weak Bear USD: 1s: {sum(short_term_weak_bear_usd_values == 1)}, 0s: {sum(short_term_weak_bear_usd_values == 0)}")
print(f"Short Term Neutral USD: 1s: {sum(short_term_neutral_usd_values == 1)}, 0s: {sum(short_term_neutral_usd_values == 0)}")
print(f"Short Term Weak Bull USD: 1s: {sum(short_term_weak_bull_usd_values == 1)}, 0s: {sum(short_term_weak_bull_usd_values == 0)}")
print(f"Short Term Strong Bull USD: 1s: {sum(short_term_strong_bull_usd_values == 1)}, 0s: {sum(short_term_strong_bull_usd_values == 0)}")

print("\nBTC Quote Trend Signals:")
print(f"Short Term Strong Bear BTC: 1s: {sum(short_term_strong_bear_btc_values == 1)}, 0s: {sum(short_term_strong_bear_btc_values == 0)}")
print(f"Short Term Weak Bear BTC: 1s: {sum(short_term_weak_bear_btc_values == 1)}, 0s: {sum(short_term_weak_bear_btc_values == 0)}")
print(f"Short Term Neutral BTC: 1s: {sum(short_term_neutral_btc_values == 1)}, 0s: {sum(short_term_neutral_btc_values == 0)}")
print(f"Short Term Weak Bull BTC: 1s: {sum(short_term_weak_bull_btc_values == 1)}, 0s: {sum(short_term_weak_bull_btc_values == 0)}")
print(f"Short Term Strong Bull BTC: 1s: {sum(short_term_strong_bull_btc_values == 1)}, 0s: {sum(short_term_strong_bull_btc_values == 0)}")

# Create our fixed calculator
calculator = BinaryForwardReturnsCalculator(periods=[3, 5, 7, 14, 30, 60])

# Calculate forward returns
data_with_returns = calculator.calculate_forward_returns(eth_data, 'close')

# Set up statistical tester
from soros_system.analysis.forward_returns.statistical_tests import StatisticalTester
tester = StatisticalTester(alpha=0.05)

# Analyze all signals
for signal_name, signal_values in [
    # USD Quote Trend Signals
    ("Short Term Strong Bear USD", short_term_strong_bear_usd_values),
    ("Short Term Weak Bear USD", short_term_weak_bear_usd_values),
    ("Short Term Neutral USD", short_term_neutral_usd_values),
    ("Short Term Weak Bull USD", short_term_weak_bull_usd_values),
    ("Short Term Strong Bull USD", short_term_strong_bull_usd_values),
    
    ("Medium Term Strong Bear USD", medium_term_strong_bear_usd_values),
    ("Medium Term Weak Bear USD", medium_term_weak_bear_usd_values),
    ("Medium Term Neutral USD", medium_term_neutral_usd_values),
    ("Medium Term Weak Bull USD", medium_term_weak_bull_usd_values),
    ("Medium Term Strong Bull USD", medium_term_strong_bull_usd_values),
    
    ("Long Term Strong Bear USD", long_term_strong_bear_usd_values),
    ("Long Term Weak Bear USD", long_term_weak_bear_usd_values),
    ("Long Term Neutral USD", long_term_neutral_usd_values),
    ("Long Term Weak Bull USD", long_term_weak_bull_usd_values),
    ("Long Term Strong Bull USD", long_term_strong_bull_usd_values),
    
    ("Overall Strong Bear USD", overall_strong_bear_usd_values),
    ("Overall Weak Bear USD", overall_weak_bear_usd_values),
    ("Overall Neutral USD", overall_neutral_usd_values),
    ("Overall Weak Bull USD", overall_weak_bull_usd_values),
    ("Overall Strong Bull USD", overall_strong_bull_usd_values),
    
    # BTC Quote Trend Signals
    ("Short Term Strong Bear BTC", short_term_strong_bear_btc_values),
    ("Short Term Weak Bear BTC", short_term_weak_bear_btc_values),
    ("Short Term Neutral BTC", short_term_neutral_btc_values),
    ("Short Term Weak Bull BTC", short_term_weak_bull_btc_values),
    ("Short Term Strong Bull BTC", short_term_strong_bull_btc_values),
    
    ("Medium Term Strong Bear BTC", medium_term_strong_bear_btc_values),
    ("Medium Term Weak Bear BTC", medium_term_weak_bear_btc_values),
    ("Medium Term Neutral BTC", medium_term_neutral_btc_values),
    ("Medium Term Weak Bull BTC", medium_term_weak_bull_btc_values),
    ("Medium Term Strong Bull BTC", medium_term_strong_bull_btc_values),
    
    ("Long Term Strong Bear BTC", long_term_strong_bear_btc_values),
    ("Long Term Weak Bear BTC", long_term_weak_bear_btc_values),
    ("Long Term Neutral BTC", long_term_neutral_btc_values),
    ("Long Term Weak Bull BTC", long_term_weak_bull_btc_values),
    ("Long Term Strong Bull BTC", long_term_strong_bull_btc_values),
    
    ("Overall Strong Bear BTC", overall_strong_bear_btc_values),
    ("Overall Weak Bear BTC", overall_weak_bear_btc_values),
    ("Overall Neutral BTC", overall_neutral_btc_values),
    ("Overall Weak Bull BTC", overall_weak_bull_btc_values),
    ("Overall Strong Bull BTC", overall_strong_bull_btc_values)
]:
    print(f"\n\n{'='*50}")
    print(f"ANALYZING {signal_name} SIGNAL")
    print(f"{'='*50}")
    
    # Get conditional returns using our fixed method that understands 0/1 signals
    conditional_returns, sample_counts = calculator.get_conditional_returns(
        data_with_returns, signal_values, min_samples=30
    )
    
    # Check if we have sufficient samples
    if sample_counts['positive'] >= 30 and sample_counts['negative'] >= 30:
        print("We have enough samples for analysis!")
        
        # Initialize results storage
        all_test_results = {}
        
        # Analyze each period
        for period in calculator.periods:
            if period in conditional_returns and 'positive' in conditional_returns[period] and 'negative' in conditional_returns[period]:
                positive_returns = conditional_returns[period]['positive']
                negative_returns = conditional_returns[period]['negative']
                all_returns = conditional_returns[period]['all']
                
                print(f"\n=== {period}-Day Forward Returns Analysis ===")
                print(f"Signal=1 samples: {len(positive_returns)}, Signal=0 samples: {len(negative_returns)}")
                
                # Run all statistical tests
                test_results = tester.run_all_tests(positive_returns, negative_returns, all_returns)
                all_test_results[period] = test_results
                
                # Evaluate effectiveness
                effectiveness = tester.evaluate_signal_effectiveness(test_results)
                
                # Print results summary
                print(f"\nEffectiveness Summary for {period}-day period:")
                print(f"Overall effective: {effectiveness['overall_effective']}")
                print(f"Confidence: {effectiveness['confidence']:.4f}")
                print(f"Effect size: {effectiveness['effect_size']:.4f}")
                
                # T-test details
                if 't_test' in test_results and test_results['t_test'].get('valid', False):
                    t_test = test_results['t_test']
                    print(f"\nT-test (mean comparison):")
                    print(f"  Significant: {t_test.get('significant', False)}")
                    print(f"  P-value: {t_test.get('p_value', 'N/A'):.4f}")
                    print(f"  Mean difference: {t_test.get('mean_difference', 0):.4f}")
                    print(f"  Effect size: {t_test.get('effect_size', 0):.4f}")
                    
                # Mann-Whitney details  
                if 'mann_whitney' in test_results and test_results['mann_whitney'].get('valid', False):
                    mw_test = test_results['mann_whitney']
                    print(f"\nMann-Whitney (distribution comparison):")
                    print(f"  Significant: {mw_test.get('significant', False)}")
                    print(f"  P-value: {mw_test.get('p_value', 'N/A'):.4f}")
                    print(f"  Median difference: {mw_test.get('median_difference', 0):.4f}")
                
                # KS-test details
                if 'ks_test' in test_results and test_results['ks_test'].get('valid', False):
                    ks_test = test_results['ks_test']
                    print(f"\nKolmogorov-Smirnov (distribution shape):")
                    print(f"  Significant: {ks_test.get('significant', False)}")
                    print(f"  P-value: {ks_test.get('p_value', 'N/A'):.4f}")
                    print(f"  Statistic: {ks_test.get('statistic', 0):.4f}")
                
                # Skew & Kurt details
                if 'skew_kurt' in test_results and test_results['skew_kurt'].get('valid', False):
                    sk_test = test_results['skew_kurt']
                    print(f"\nSkewness & Kurtosis:")
                    print(f"  Signal=1 skew: {sk_test.get('positive_skew', 0):.4f}")
                    print(f"  Signal=0 skew: {sk_test.get('negative_skew', 0):.4f}")
                    print(f"  Signal=1 kurt: {sk_test.get('positive_kurt', 0):.4f}")
                    print(f"  Signal=0 kurt: {sk_test.get('negative_kurt', 0):.4f}")
        
        # Calculate overall effectiveness
        effective_periods = 0
        total_periods = len(all_test_results)
        effect_sizes = []
        confidences = []
        mean_diffs = []
        
        for period, test_results in all_test_results.items():
            effectiveness = tester.evaluate_signal_effectiveness(test_results)
            if effectiveness.get('overall_effective', False):
                effective_periods += 1
                effect_sizes.append(abs(effectiveness.get('effect_size', 0.0)))
                confidences.append(effectiveness.get('confidence', 0.0))
                
                if 't_test' in test_results and test_results['t_test'].get('valid', False):
                    mean_diffs.append(test_results['t_test'].get('mean_difference', 0))
        
        # Calculate summary metrics
        effectiveness_ratio = effective_periods / total_periods if total_periods > 0 else 0.0
        avg_effect_size = np.mean(effect_sizes) if effect_sizes else 0.0
        avg_confidence = np.mean(confidences) if confidences else 0.0
        avg_mean_diff = np.mean(mean_diffs) if mean_diffs else 0.0
        
        # Determine overall effectiveness
        overall_effective = effectiveness_ratio >= 0.5 and avg_effect_size > 0.15
        
        # Calculate weight
        if overall_effective:
            weight = (avg_effect_size * 0.5) + (avg_confidence * 0.3) + (abs(avg_mean_diff) * 100 * 0.2)
        else:
            weight = (avg_effect_size * 0.25) + (avg_confidence * 0.15) + (abs(avg_mean_diff) * 100 * 0.1)
            if effectiveness_ratio < 0.3 or avg_effect_size < 0.1:
                weight = 0.0
        
        # Print overall summary
        print("\n=== OVERALL SUMMARY ===")
        print(f"Signal: {signal_name}")
        print(f"Effective periods: {effective_periods}/{total_periods}")
        print(f"Effectiveness ratio: {effectiveness_ratio:.4f}")
        print(f"Average effect size: {avg_effect_size:.4f}")
        print(f"Average confidence: {avg_confidence:.4f}")
        print(f"Average mean difference: {avg_mean_diff:.4f}")
        print(f"Overall effectiveness: {overall_effective}")
        print(f"Weight: {weight:.4f}")
        
    else:
        print("Not enough samples for analysis")

# Now generate all the plots after all the analysis is printed
for signal_name, signal_values in [
    # USD Quote Trend Signals
    ("Short Term Strong Bear USD", short_term_strong_bear_usd_values),
    ("Short Term Weak Bear USD", short_term_weak_bear_usd_values),
    ("Short Term Neutral USD", short_term_neutral_usd_values),
    ("Short Term Weak Bull USD", short_term_weak_bull_usd_values),
    ("Short Term Strong Bull USD", short_term_strong_bull_usd_values),
    
    ("Medium Term Strong Bear USD", medium_term_strong_bear_usd_values),
    ("Medium Term Weak Bear USD", medium_term_weak_bear_usd_values),
    ("Medium Term Neutral USD", medium_term_neutral_usd_values),
    ("Medium Term Weak Bull USD", medium_term_weak_bull_usd_values),
    ("Medium Term Strong Bull USD", medium_term_strong_bull_usd_values),
    
    ("Long Term Strong Bear USD", long_term_strong_bear_usd_values),
    ("Long Term Weak Bear USD", long_term_weak_bear_usd_values),
    ("Long Term Neutral USD", long_term_neutral_usd_values),
    ("Long Term Weak Bull USD", long_term_weak_bull_usd_values),
    ("Long Term Strong Bull USD", long_term_strong_bull_usd_values),
    
    ("Overall Strong Bear USD", overall_strong_bear_usd_values),
    ("Overall Weak Bear USD", overall_weak_bear_usd_values),
    ("Overall Neutral USD", overall_neutral_usd_values),
    ("Overall Weak Bull USD", overall_weak_bull_usd_values),
    ("Overall Strong Bull USD", overall_strong_bull_usd_values),
    
    # BTC Quote Trend Signals
    ("Short Term Strong Bear BTC", short_term_strong_bear_btc_values),
    ("Short Term Weak Bear BTC", short_term_weak_bear_btc_values),
    ("Short Term Neutral BTC", short_term_neutral_btc_values),
    ("Short Term Weak Bull BTC", short_term_weak_bull_btc_values),
    ("Short Term Strong Bull BTC", short_term_strong_bull_btc_values),
    
    ("Medium Term Strong Bear BTC", medium_term_strong_bear_btc_values),
    ("Medium Term Weak Bear BTC", medium_term_weak_bear_btc_values),
    ("Medium Term Neutral BTC", medium_term_neutral_btc_values),
    ("Medium Term Weak Bull BTC", medium_term_weak_bull_btc_values),
    ("Medium Term Strong Bull BTC", medium_term_strong_bull_btc_values),
    
    ("Long Term Strong Bear BTC", long_term_strong_bear_btc_values),
    ("Long Term Weak Bear BTC", long_term_weak_bear_btc_values),
    ("Long Term Neutral BTC", long_term_neutral_btc_values),
    ("Long Term Weak Bull BTC", long_term_weak_bull_btc_values),
    ("Long Term Strong Bull BTC", long_term_strong_bull_btc_values),
    
    ("Overall Strong Bear BTC", overall_strong_bear_btc_values),
    ("Overall Weak Bear BTC", overall_weak_bear_btc_values),
    ("Overall Neutral BTC", overall_neutral_btc_values),
    ("Overall Weak Bull BTC", overall_weak_bull_btc_values),
    ("Overall Strong Bull BTC", overall_strong_bull_btc_values)
]:
    print(f"\n\n{'='*50}")
    print(f"GENERATING PLOTS FOR {signal_name} SIGNAL")
    print(f"{'='*50}")
    
    # Get conditional returns
    conditional_returns, sample_counts = calculator.get_conditional_returns(
        data_with_returns, signal_values, min_samples=30
    )
    
    # Check if we have sufficient samples
    if sample_counts['positive'] >= 30 and sample_counts['negative'] >= 30:
        # Analyze each period
        for period in calculator.periods:
            if period in conditional_returns and 'positive' in conditional_returns[period] and 'negative' in conditional_returns[period]:
                positive_returns = conditional_returns[period]['positive']
                negative_returns = conditional_returns[period]['negative']
                
                # Plot distributions
                plt.figure(figsize=(12, 6))
                sns.kdeplot(data=positive_returns, label='Signal = 1', color='green', alpha=0.6)
                sns.kdeplot(data=negative_returns, label='Signal = 0', color='red', alpha=0.6)
                
                # Add vertical lines for means and medians
                plt.axvline(positive_returns.mean(), color='green', linestyle='--', alpha=0.8, label='Mean (Signal=1)')
                plt.axvline(negative_returns.mean(), color='red', linestyle='--', alpha=0.8, label='Mean (Signal=0)')
                plt.axvline(positive_returns.median(), color='green', linestyle=':', alpha=0.8, label='Median (Signal=1)')
                plt.axvline(negative_returns.median(), color='red', linestyle=':', alpha=0.8, label='Median (Signal=0)')
                
                # Add title and labels
                plt.title(f'{signal_name} - {period}-Day Forward Return Distributions')
                plt.xlabel('Forward Returns')
                plt.ylabel('Density')
                plt.legend()
                
                # Add statistical annotations
                stats_text = f"""
                Signal=1 Mean: {positive_returns.mean():.4f}
                Signal=0 Mean: {negative_returns.mean():.4f}
                Signal=1 Median: {positive_returns.median():.4f}
                Signal=0 Median: {negative_returns.median():.4f}
                Signal=1 Std: {positive_returns.std():.4f}
                Signal=0 Std: {negative_returns.std():.4f}
                Signal=1 Skew: {stats.skew(positive_returns):.4f}
                Signal=0 Skew: {stats.skew(negative_returns):.4f}
                Signal=1 Count: {len(positive_returns)}
                Signal=0 Count: {len(negative_returns)}
                """
                plt.text(0.02, 0.98, stats_text,
                         transform=plt.gca().transAxes,
                         verticalalignment='top',
                         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                
                plt.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()
    else:
        print("Not enough samples for plots")

In [ ]:
asset = 'aave'

# Get data for Ethereum
eth_data = analyzer.get_asset_raw_data(asset)
print(f"Testing with {len(eth_data)} days of {asset} data")

# Create instances of the signals
markov_low_volatility = MarkovLowVolatilitySignal()
markov_high_volatility = MarkovHighVolatilitySignal()
ssr_risk_on = SSR_RiskOn({})  # Pass empty dict instead of None
ssr_risk_off = SSR_RiskOff({})  # Pass empty dict instead of None

# Calculate the signals
markov_low_volatility_values = markov_low_volatility.calculate(eth_data, asset)
markov_high_volatility_values = markov_high_volatility.calculate(eth_data, asset)
ssr_risk_on_values = ssr_risk_on.calculate(eth_data, asset)
ssr_risk_off_values = ssr_risk_off.calculate(eth_data, asset)

# Print signal counts
print("\nVolatility and Risk Signals:")
print(f"Markov Low Volatility: 1s: {sum(markov_low_volatility_values == 1)}, 0s: {sum(markov_low_volatility_values == 0)}")
print(f"Markov High Volatility: 1s: {sum(markov_high_volatility_values == 1)}, 0s: {sum(markov_high_volatility_values == 0)}")
print(f"SSR Risk On: 1s: {sum(ssr_risk_on_values == 1)}, 0s: {sum(ssr_risk_on_values == 0)}")
print(f"SSR Risk Off: 1s: {sum(ssr_risk_off_values == 1)}, 0s: {sum(ssr_risk_off_values == 0)}")

# Create our fixed calculator
calculator = BinaryForwardReturnsCalculator(periods=[7, 14, 30, 60])

# Calculate forward returns
data_with_returns = calculator.calculate_forward_returns(eth_data, 'close')

# Set up statistical tester
from soros_system.analysis.forward_returns.statistical_tests import StatisticalTester
tester = StatisticalTester(alpha=0.05)

# Analyze all signals
for signal_name, signal_values in [
    ("Markov Low Volatility", markov_low_volatility_values),
    ("Markov High Volatility", markov_high_volatility_values),
    ("SSR Risk On", ssr_risk_on_values),
    ("SSR Risk Off", ssr_risk_off_values)
]:
    print(f"\n\n{'='*50}")
    print(f"ANALYZING {signal_name} SIGNAL")
    print(f"{'='*50}")
    
    # Get conditional returns using our fixed method that understands 0/1 signals
    conditional_returns, sample_counts = calculator.get_conditional_returns(
        data_with_returns, signal_values, min_samples=30
    )
    
    # Check if we have sufficient samples
    if sample_counts['positive'] >= 30 and sample_counts['negative'] >= 30:
        print("We have enough samples for analysis!")
        
        # Initialize results storage
        all_test_results = {}
        
        # Analyze each period
        for period in calculator.periods:
            if period in conditional_returns and 'positive' in conditional_returns[period] and 'negative' in conditional_returns[period]:
                positive_returns = conditional_returns[period]['positive']
                negative_returns = conditional_returns[period]['negative']
                all_returns = conditional_returns[period]['all']
                
                print(f"\n=== {period}-Day Forward Returns Analysis ===")
                print(f"Signal=1 samples: {len(positive_returns)}, Signal=0 samples: {len(negative_returns)}")
                
                # Run all statistical tests
                test_results = tester.run_all_tests(positive_returns, negative_returns, all_returns)
                all_test_results[period] = test_results
                
                # Evaluate effectiveness
                effectiveness = tester.evaluate_signal_effectiveness(test_results)
                
                # Print results summary
                print(f"\nEffectiveness Summary for {period}-day period:")
                print(f"Overall effective: {effectiveness['overall_effective']}")
                print(f"Confidence: {effectiveness['confidence']:.4f}")
                print(f"Effect size: {effectiveness['effect_size']:.4f}")
                
                # T-test details
                if 't_test' in test_results and test_results['t_test'].get('valid', False):
                    t_test = test_results['t_test']
                    print(f"\nT-test (mean comparison):")
                    print(f"  Significant: {t_test.get('significant', False)}")
                    print(f"  P-value: {t_test.get('p_value', 'N/A'):.4f}")
                    print(f"  Mean difference: {t_test.get('mean_difference', 0):.4f}")
                    print(f"  Effect size: {t_test.get('effect_size', 0):.4f}")
                    
                # Mann-Whitney details  
                if 'mann_whitney' in test_results and test_results['mann_whitney'].get('valid', False):
                    mw_test = test_results['mann_whitney']
                    print(f"\nMann-Whitney (distribution comparison):")
                    print(f"  Significant: {mw_test.get('significant', False)}")
                    print(f"  P-value: {mw_test.get('p_value', 'N/A'):.4f}")
                    print(f"  Median difference: {mw_test.get('median_difference', 0):.4f}")
                
                # KS-test details
                if 'ks_test' in test_results and test_results['ks_test'].get('valid', False):
                    ks_test = test_results['ks_test']
                    print(f"\nKolmogorov-Smirnov (distribution shape):")
                    print(f"  Significant: {ks_test.get('significant', False)}")
                    print(f"  P-value: {ks_test.get('p_value', 'N/A'):.4f}")
                    print(f"  Statistic: {ks_test.get('statistic', 0):.4f}")
                
                # Skew & Kurt details
                if 'skew_kurt' in test_results and test_results['skew_kurt'].get('valid', False):
                    sk_test = test_results['skew_kurt']
                    print(f"\nSkewness & Kurtosis:")
                    print(f"  Signal=1 skew: {sk_test.get('positive_skew', 0):.4f}")
                    print(f"  Signal=0 skew: {sk_test.get('negative_skew', 0):.4f}")
                    print(f"  Signal=1 kurt: {sk_test.get('positive_kurt', 0):.4f}")
                    print(f"  Signal=0 kurt: {sk_test.get('negative_kurt', 0):.4f}")
        
        # Calculate overall effectiveness
        effective_periods = 0
        total_periods = len(all_test_results)
        effect_sizes = []
        confidences = []
        mean_diffs = []
        
        for period, test_results in all_test_results.items():
            effectiveness = tester.evaluate_signal_effectiveness(test_results)
            if effectiveness.get('overall_effective', False):
                effective_periods += 1
                effect_sizes.append(abs(effectiveness.get('effect_size', 0.0)))
                confidences.append(effectiveness.get('confidence', 0.0))
                
                if 't_test' in test_results and test_results['t_test'].get('valid', False):
                    mean_diffs.append(test_results['t_test'].get('mean_difference', 0))
        
        # Calculate summary metrics
        effectiveness_ratio = effective_periods / total_periods if total_periods > 0 else 0.0
        avg_effect_size = np.mean(effect_sizes) if effect_sizes else 0.0
        avg_confidence = np.mean(confidences) if confidences else 0.0
        avg_mean_diff = np.mean(mean_diffs) if mean_diffs else 0.0
        
        # Determine overall effectiveness
        overall_effective = effectiveness_ratio >= 0.5 and avg_effect_size > 0.15
        
        # Calculate weight
        if overall_effective:
            weight = (avg_effect_size * 0.5) + (avg_confidence * 0.3) + (abs(avg_mean_diff) * 100 * 0.2)
        else:
            weight = (avg_effect_size * 0.25) + (avg_confidence * 0.15) + (abs(avg_mean_diff) * 100 * 0.1)
            if effectiveness_ratio < 0.3 or avg_effect_size < 0.1:
                weight = 0.0
        
        # Print overall summary
        print("\n=== OVERALL SUMMARY ===")
        print(f"Signal: {signal_name}")
        print(f"Effective periods: {effective_periods}/{total_periods}")
        print(f"Effectiveness ratio: {effectiveness_ratio:.4f}")
        print(f"Average effect size: {avg_effect_size:.4f}")
        print(f"Average confidence: {avg_confidence:.4f}")
        print(f"Average mean difference: {avg_mean_diff:.4f}")
        print(f"Overall effectiveness: {overall_effective}")
        print(f"Weight: {weight:.4f}")
        
    else:
        print("Not enough samples for analysis")

# Now generate all the plots after all the analysis is printed
for signal_name, signal_values in [
    ("Markov Low Volatility", markov_low_volatility_values),
    ("Markov High Volatility", markov_high_volatility_values),
    ("SSR Risk On", ssr_risk_on_values),
    ("SSR Risk Off", ssr_risk_off_values)
]:
    print(f"\n\n{'='*50}")
    print(f"GENERATING PLOTS FOR {signal_name} SIGNAL")
    print(f"{'='*50}")
    
    # Get conditional returns
    conditional_returns, sample_counts = calculator.get_conditional_returns(
        data_with_returns, signal_values, min_samples=30
    )
    
    # Check if we have sufficient samples
    if sample_counts['positive'] >= 30 and sample_counts['negative'] >= 30:
        # Analyze each period
        for period in calculator.periods:
            if period in conditional_returns and 'positive' in conditional_returns[period] and 'negative' in conditional_returns[period]:
                positive_returns = conditional_returns[period]['positive']
                negative_returns = conditional_returns[period]['negative']
                
                # Plot distributions
                plt.figure(figsize=(12, 6))
                sns.kdeplot(data=positive_returns, label='Signal = 1', color='green', alpha=0.6)
                sns.kdeplot(data=negative_returns, label='Signal = 0', color='red', alpha=0.6)
                
                # Add vertical lines for means and medians
                plt.axvline(positive_returns.mean(), color='green', linestyle='--', alpha=0.8, label='Mean (Signal=1)')
                plt.axvline(negative_returns.mean(), color='red', linestyle='--', alpha=0.8, label='Mean (Signal=0)')
                plt.axvline(positive_returns.median(), color='green', linestyle=':', alpha=0.8, label='Median (Signal=1)')
                plt.axvline(negative_returns.median(), color='red', linestyle=':', alpha=0.8, label='Median (Signal=0)')
                
                # Add title and labels
                plt.title(f'{signal_name} - {period}-Day Forward Return Distributions')
                plt.xlabel('Forward Returns')
                plt.ylabel('Density')
                plt.legend()
                
                # Add statistical annotations
                stats_text = f"""
                Signal=1 Mean: {positive_returns.mean():.4f}
                Signal=0 Mean: {negative_returns.mean():.4f}
                Signal=1 Median: {positive_returns.median():.4f}
                Signal=0 Median: {negative_returns.median():.4f}
                Signal=1 Std: {positive_returns.std():.4f}
                Signal=0 Std: {negative_returns.std():.4f}
                Signal=1 Skew: {stats.skew(positive_returns):.4f}
                Signal=0 Skew: {stats.skew(negative_returns):.4f}
                Signal=1 Count: {len(positive_returns)}
                Signal=0 Count: {len(negative_returns)}
                """
                plt.text(0.02, 0.98, stats_text,
                         transform=plt.gca().transAxes,
                         verticalalignment='top',
                         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                
                plt.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()
    else:
        print("Not enough samples for plots")